# 18 — Evaluation

Measures the fine-tuned model across five dimensions:
1. **Syntax pass rate** — generated ARO code passes `aro check`
2. **Debugging accuracy** — fixed code passes, buggy code fails
3. **Token overlap** — Q&A / explanation quality vs reference answers
4. **Hallucination rate** — model invents non-existent ARO verbs
5. **FIM accuracy** — fill-in-the-middle exact match

Also compares the fine-tuned model against the base model on the same prompts.

**Input:**  `../data/05_dataset/test.jsonl`  (from notebook 15)
            `../data/02_knowledge/knowledge.json`  (from notebook 02)
            `../models/round_0/adapter/`  (from notebook 16)
**Output:** `../data/07_eval/report.json`  +  printed summary
            `../data/07_eval/details.jsonl`

In [1]:
import json
import re
import sys
import importlib
import subprocess
import tempfile
import platform
from pathlib import Path
from collections import defaultdict, Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
import config; importlib.reload(config); from config import *

DATA_DIR    = DATA_ROOT / '05_dataset'
KB_DIR      = DATA_ROOT / '02_knowledge'
ADAPTER_DIR = FINETUNE_MODELS_DIR / 'round_0' / 'adapter'
EVAL_DIR    = DATA_ROOT / '07_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL  = 'mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit'
ROUND       = 0
MAX_EVAL    = 200   # number of test samples to evaluate (set to None for all)
MAX_TOKENS  = 512

print(f'Test data:   {DATA_DIR / "test.jsonl"}')
print(f'Adapter:     {ADAPTER_DIR}')
print(f'Eval output: {EVAL_DIR}')

TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Test data:   /Users/kris/Projects/ARO/ARO-Train/Train/data/05_dataset/test.jsonl
Adapter:     /Users/kris/Projects/ARO/ARO-Train/Train/models/finetune/round_0/adapter
Eval output: /Users/kris/Projects/ARO/ARO-Train/Train/data/07_eval


## Load test set + knowledge base

In [2]:
test_data = []
with open(DATA_DIR / 'test.jsonl') as f:
    for line in f:
        if line.strip():
            test_data.append(json.loads(line))

if MAX_EVAL:
    import random
    random.seed(0)
    test_data = random.sample(test_data, min(MAX_EVAL, len(test_data)))

print(f'Test samples: {len(test_data)}')
print('Distribution:', dict(Counter(s['task_type'] for s in test_data)))

# Load known verbs for hallucination check
with open(KB_DIR / 'knowledge.json') as f:
    kb = json.load(f)

known_verbs = set()
for action in kb['actions']:
    for v in action['verbs']:
        known_verbs.add(v.lower())

print(f'Known ARO verbs: {len(known_verbs)}')

Test samples: 39
Distribution: {'syntax_qa': 24, 'code_generation': 14, 'code_explanation': 1}
Known ARO verbs: 98


## Load models

In [3]:
from mlx_lm import load, generate as _mlx_generate

print(f'Loading fine-tuned model ({BASE_MODEL} + adapter)...')
ft_model, ft_tokenizer = load(BASE_MODEL, adapter_path=str(ADAPTER_DIR))
print('Fine-tuned model ready.')

print(f'Loading base model (no adapter)...')
base_model, base_tokenizer = load(BASE_MODEL)
print('Base model ready.')

/Users/kris/Projects/ARO/ARO-Train/Train/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading fine-tuned model (mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit + adapter)...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 123847.56it/s]


Fine-tuned model ready.
Loading base model (no adapter)...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 347594.25it/s]


Base model ready.


## Generation + validation helpers

In [4]:
def build_prompt(sample, tokenizer):
    """Reconstruct the inference prompt (system + user turns only)."""
    msgs = sample['messages'][:2]  # system + user
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(model, tokenizer, prompt, max_tokens=MAX_TOKENS):
    return _mlx_generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)

def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]

def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return None

def token_overlap(generated, reference):
    """F1-style token overlap between generated and reference text."""
    gen_tokens = set(generated.lower().split())
    ref_tokens = set(reference.lower().split())
    if not ref_tokens:
        return 0.0
    precision = len(gen_tokens & ref_tokens) / max(1, len(gen_tokens))
    recall    = len(gen_tokens & ref_tokens) / max(1, len(ref_tokens))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def hallucination_score(text):
    """Fraction of verbs in generated ARO code that are NOT in known_verbs."""
    blocks = extract_aro_blocks(text)
    if not blocks:
        return None
    verbs_found = re.findall(r'^\s+([A-Z][a-z]+)', '\n'.join(blocks), re.MULTILINE)
    if not verbs_found:
        return None
    unknown = [v for v in verbs_found if v.lower() not in known_verbs]
    return len(unknown) / len(verbs_found)

## Run evaluation

In [5]:
results = defaultdict(lambda: {
    'total': 0, 'ft_syntax': 0, 'base_syntax': 0,
    'ft_overlap': [], 'base_overlap': [],
    'ft_hallucination': [], 'base_hallucination': [],
    'ft_fim_exact': 0, 'base_fim_exact': 0,
    'debug_fix_pass': 0, 'debug_buggy_fail': 0,
})

detail_rows = []

for i, sample in enumerate(test_data):
    task   = sample['task_type']
    ref    = sample['messages'][2]['content']
    prompt = build_prompt(sample, ft_tokenizer)

    try:
        ft_out   = generate(ft_model,   ft_tokenizer,   prompt)
        base_out = generate(base_model, base_tokenizer, prompt)
    except Exception as e:
        print(f'  ! generate error at {i}: {e}')
        continue

    r = results[task]
    r['total'] += 1

    # --- Syntax pass rate (code tasks) ---
    if task in ('code_generation', 'translation', 'fim'):
        for label, out in [('ft', ft_out), ('base', base_out)]:
            blocks = extract_aro_blocks(out) or [out]
            code   = '\n\n'.join(blocks)
            passed = aro_check(code)
            if passed is True:
                r[f'{label}_syntax'] += 1

    # --- FIM exact match ---
    if task == 'fim':
        middle = sample.get('middle', '').strip()
        if ft_out.strip() == middle:
            r['ft_fim_exact'] += 1
        if base_out.strip() == middle:
            r['base_fim_exact'] += 1

    # --- Debugging: fix must pass, buggy must fail ---
    if task == 'debugging':
        fix_m = re.search(r'## Fix\s*```aro\n(.*?)```', ft_out, re.DOTALL)
        if fix_m:
            if aro_check(fix_m.group(1).strip()) is True:
                r['debug_fix_pass'] += 1
        buggy_m = re.search(r'## Buggy Code\s*```aro\n(.*?)```', ft_out, re.DOTALL)
        if buggy_m:
            if aro_check(buggy_m.group(1).strip()) is False:
                r['debug_buggy_fail'] += 1

    # --- Token overlap (explanation / QA / architecture) ---
    if task in ('code_explanation', 'syntax_qa', 'architecture'):
        r['ft_overlap'].append(token_overlap(ft_out, ref))
        r['base_overlap'].append(token_overlap(base_out, ref))

    # --- Hallucination rate ---
    for label, out in [('ft', ft_out), ('base', base_out)]:
        score = hallucination_score(out)
        if score is not None:
            r[f'{label}_hallucination'].append(score)

    detail_rows.append({
        'task': task, 'ft_output': ft_out[:500], 'base_output': base_out[:500], 'reference': ref[:500],
    })

    if (i + 1) % 20 == 0:
        print(f'  Evaluated {i+1}/{len(test_data)}')

print('Evaluation complete.')

  Evaluated 20/39
Evaluation complete.


## Print report

In [6]:
def avg(lst):
    return sum(lst) / len(lst) if lst else 0.0

print('\n' + '=' * 70)
print(f'{"METRIC":<35} {"FINE-TUNED":>12} {"BASE":>12}')
print('=' * 70)

report = {}

for task, r in sorted(results.items()):
    n = r['total']
    if n == 0:
        continue

    report[task] = {}
    print(f'\n  [{task}]  n={n}')

    if r['ft_syntax'] or r['base_syntax']:
        ft_rate   = r['ft_syntax'] / n
        base_rate = r['base_syntax'] / n
        print(f'  {"Syntax pass rate":<33} {ft_rate:>11.1%} {base_rate:>11.1%}')
        report[task]['ft_syntax_pass_rate']   = round(ft_rate, 3)
        report[task]['base_syntax_pass_rate'] = round(base_rate, 3)

    if r['ft_overlap']:
        ft_ov   = avg(r['ft_overlap'])
        base_ov = avg(r['base_overlap'])
        print(f'  {"Token overlap (F1)":<33} {ft_ov:>11.2f} {base_ov:>11.2f}')
        report[task]['ft_token_overlap']   = round(ft_ov, 3)
        report[task]['base_token_overlap'] = round(base_ov, 3)

    if r['ft_hallucination']:
        ft_h   = avg(r['ft_hallucination'])
        base_h = avg(r['base_hallucination'])
        print(f'  {"Hallucination rate (lower=better)":<33} {ft_h:>11.1%} {base_h:>11.1%}')
        report[task]['ft_hallucination_rate']   = round(ft_h, 3)
        report[task]['base_hallucination_rate'] = round(base_h, 3)

    if task == 'fim' and (r['ft_fim_exact'] or r['base_fim_exact']):
        ft_ex   = r['ft_fim_exact'] / n
        base_ex = r['base_fim_exact'] / n
        print(f'  {"FIM exact match":<33} {ft_ex:>11.1%} {base_ex:>11.1%}')
        report[task]['ft_fim_exact_match']   = round(ft_ex, 3)
        report[task]['base_fim_exact_match'] = round(base_ex, 3)

    if task == 'debugging' and r['debug_fix_pass']:
        fix_rate = r['debug_fix_pass'] / n
        print(f'  {"Fix-passes-check rate":<33} {fix_rate:>11.1%}')
        report[task]['debug_fix_pass_rate'] = round(fix_rate, 3)

print('\n' + '=' * 70)


METRIC                                FINE-TUNED         BASE

  [code_explanation]  n=1
  Token overlap (F1)                       0.46        0.29

  [code_generation]  n=14
  Syntax pass rate                        35.7%        7.1%
  Hallucination rate (lower=better)       30.8%       26.0%

  [syntax_qa]  n=24
  Token overlap (F1)                       0.37        0.19
  Hallucination rate (lower=better)       19.4%       24.1%



## Save report

In [7]:
with open(EVAL_DIR / 'report.json', 'w') as f:
    json.dump(report, f, indent=2)

with open(EVAL_DIR / 'details.jsonl', 'w') as f:
    for row in detail_rows:
        f.write(json.dumps(row) + '\n')

print(f'\nReport saved: {EVAL_DIR / "report.json"}')
print(f'Details saved: {EVAL_DIR / "details.jsonl"}')


Report saved: /Users/kris/Projects/ARO/ARO-Train/Train/data/07_eval/report.json
Details saved: /Users/kris/Projects/ARO/ARO-Train/Train/data/07_eval/details.jsonl


In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)

_metric_map = {
    'ft_syntax_pass_rate':    ('syntax_pass_rate',    'base_syntax_pass_rate'),
    'ft_token_overlap':       ('token_overlap_f1',    'base_token_overlap'),
    'ft_hallucination_rate':  ('hallucination_rate',  'base_hallucination_rate'),
    'ft_fim_exact_match':     ('fim_exact_match',     'base_fim_exact_match'),
    'debug_fix_pass_rate':    ('debug_fix_pass_rate', None),
}

with open(_run_dir / '19_evaluation.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_type', 'metric', 'fine_tuned_score', 'base_score'])
    for task in sorted(report):
        for ft_key, (metric_name, base_key) in _metric_map.items():
            if ft_key in report[task]:
                ft_val = report[task][ft_key]
                base_val = report[task].get(base_key, '') if base_key else ''
                w.writerow([task, metric_name, ft_val, base_val])

print(f'Saved: {_run_dir / "19_evaluation.csv"}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '19_evaluation.png'

# Collect metrics for plotting
_tasks = sorted(t for t in report if report[t])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel 1: Syntax pass rate (FT vs Base) ───────────────────────────
ax = axes[0]
_syn_tasks = [t for t in _tasks if 'ft_syntax_pass_rate' in report[t]]
if _syn_tasks:
    x = np.arange(len(_syn_tasks))
    w = 0.35
    ft_vals = [report[t]['ft_syntax_pass_rate'] * 100 for t in _syn_tasks]
    base_vals = [report[t]['base_syntax_pass_rate'] * 100 for t in _syn_tasks]
    b1 = ax.bar(x - w/2, ft_vals, w, label='Fine-tuned', color='#2ecc71', edgecolor='white')
    b2 = ax.bar(x + w/2, base_vals, w, label='Base', color='#e74c3c', edgecolor='white')
    ax.bar_label(b1, fmt='%.0f%%', padding=3, fontsize=9)
    ax.bar_label(b2, fmt='%.0f%%', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace('_', '\n') for t in _syn_tasks], fontsize=8)
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'no syntax data', ha='center', va='center', transform=ax.transAxes)
ax.set_ylabel('Pass rate (%)')
ax.set_title('Syntax Pass Rate', fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

# ── Panel 2: Token overlap F1 ────────────────────────────────────────
ax = axes[1]
_ov_tasks = [t for t in _tasks if 'ft_token_overlap' in report[t]]
if _ov_tasks:
    x = np.arange(len(_ov_tasks))
    w = 0.35
    ft_vals = [report[t]['ft_token_overlap'] for t in _ov_tasks]
    base_vals = [report[t]['base_token_overlap'] for t in _ov_tasks]
    b1 = ax.bar(x - w/2, ft_vals, w, label='Fine-tuned', color='#3498db', edgecolor='white')
    b2 = ax.bar(x + w/2, base_vals, w, label='Base', color='#e67e22', edgecolor='white')
    ax.bar_label(b1, fmt='%.2f', padding=3, fontsize=9)
    ax.bar_label(b2, fmt='%.2f', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace('_', '\n') for t in _ov_tasks], fontsize=8)
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'no overlap data', ha='center', va='center', transform=ax.transAxes)
ax.set_ylabel('Token Overlap (F1)')
ax.set_title('Answer Quality', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# ── Panel 3: Hallucination rate ───────────────────────────────────────
ax = axes[2]
_hal_tasks = [t for t in _tasks if 'ft_hallucination_rate' in report[t]]
if _hal_tasks:
    x = np.arange(len(_hal_tasks))
    w = 0.35
    ft_vals = [report[t]['ft_hallucination_rate'] * 100 for t in _hal_tasks]
    base_vals = [report[t]['base_hallucination_rate'] * 100 for t in _hal_tasks]
    b1 = ax.bar(x - w/2, ft_vals, w, label='Fine-tuned', color='#9b59b6', edgecolor='white')
    b2 = ax.bar(x + w/2, base_vals, w, label='Base', color='#95a5a6', edgecolor='white')
    ax.bar_label(b1, fmt='%.0f%%', padding=3, fontsize=9)
    ax.bar_label(b2, fmt='%.0f%%', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace('_', '\n') for t in _hal_tasks], fontsize=8)
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'no hallucination data', ha='center', va='center', transform=ax.transAxes)
ax.set_ylabel('Hallucination rate (%)')
ax.set_title('Hallucination Rate (lower = better)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

fig.suptitle(
    f'Evaluation — Fine-tuned vs Base  ·  {len(test_data)} test samples',
    fontsize=14, fontweight='bold', y=1.02,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


## Inspect failures

In [8]:
# Show 5 cases where fine-tuned model produced invalid ARO code
print('=== Sample failures (ft model produced invalid ARO) ===\n')
shown = 0
for row in detail_rows:
    if row['task'] not in ('code_generation', 'translation'):
        continue
    blocks = extract_aro_blocks(row['ft_output'])
    if blocks and aro_check('\n\n'.join(blocks)) is False:
        print(f'Task: {row["task"]}')
        print(f'Generated:\n{row["ft_output"][:400]}\n')
        shown += 1
    if shown >= 5:
        break

if shown == 0:
    print('No failures found in sample set.')

=== Sample failures (ft model produced invalid ARO) ===

Task: code_generation
Generated:
## main.aro
```aro
(Application-Start: ZipService) {
    Log "Testing zip functionality..." to the <console>.
    Log "Compressing files into archive.zip..." to the <console>.
    Zip the <content> with <archive.zip>.
    Return an <OK: status> for the <result>.
}

(Application-End: Success) {
    Log "Zip service completed successfully." to the <console>.
    Return an <OK: status> for the <resul

Task: code_generation
Generated:
## main.aro
```aro
(Application: Main) {
    Import the <features> from the <auth: features>.
    Import the <types> from the <payment: types>.
    Return an <OK: status> for the <import>.
}
```

Task: code_generation
Generated:
```aro
(Feature Set: Execute Command) {
    Extract the <command> from the <request: body>.
    Execute the <output> on the <shell> with <command>.
    When <exit-code> = 0 {
        Return an <OK: status> with <output>.
    }
    When <exit-code

## Summary

In [9]:
print('=' * 65)
print('notebook 18 — EVALUATION SUMMARY')
print('=' * 65)

def _avg(lst): return sum(lst) / len(lst) if lst else None

print(f'\nTest set: {len(test_data)} samples')
task_dist = Counter(s['task_type'] for s in test_data)
for t, n in task_dist.most_common():
    print(f'  {t:25s}: {n}')

print(f'\nMetrics (fine-tuned vs base):')
print(f'  {"Task":<22} {"Metric":<28} {"FT":>8} {"Base":>8} {"Delta":>8}')
print(f'  {"-"*22} {"-"*28} {"-"*8} {"-"*8} {"-"*8}')

_overall_ft_overlap   = []
_overall_base_overlap = []
_overall_ft_syntax    = []
_overall_base_syntax  = []

for task, r in sorted(report.items()):
    if 'ft_token_overlap' in r:
        ft  = r['ft_token_overlap']
        b   = r['base_token_overlap']
        d   = ft - b
        flag = '  ⚠  regressed' if d < -0.05 else ('  ✓' if d > 0.02 else '')
        print(f'  {task:<22} {"token_overlap (F1)":<28} {ft:>8.3f} {b:>8.3f} {d:>+8.3f}{flag}')
        _overall_ft_overlap.append(ft)
        _overall_base_overlap.append(b)
    if 'ft_syntax_pass_rate' in r:
        ft  = r['ft_syntax_pass_rate']
        b   = r['base_syntax_pass_rate']
        d   = ft - b
        flag = '  ⚠  regressed' if d < -0.05 else ('  ✓' if d > 0.05 else '')
        print(f'  {task:<22} {"syntax_pass_rate":<28} {ft:>7.1%} {b:>7.1%} {d:>+7.1%}{flag}')
        _overall_ft_syntax.append(ft)
        _overall_base_syntax.append(b)
    if 'ft_hallucination_rate' in r:
        ft  = r['ft_hallucination_rate']
        b   = r['base_hallucination_rate']
        d   = ft - b
        flag = '  ⚠  more hallucinations' if d > 0.05 else ('  ✓  fewer hallucinations' if d < -0.05 else '')
        print(f'  {task:<22} {"hallucination_rate (↓)":<28} {ft:>7.1%} {b:>7.1%} {d:>+7.1%}{flag}')

print(f'\nOverall:')
if _overall_ft_overlap:
    ft_ov  = _avg(_overall_ft_overlap)
    b_ov   = _avg(_overall_base_overlap)
    print(f'  Avg token overlap:  FT={ft_ov:.3f}  Base={b_ov:.3f}  Δ={ft_ov-b_ov:+.3f}')
    if ft_ov < b_ov - 0.05:
        print('  🚨  Fine-tuned model is WORSE than base on token overlap.')
        print('      This confirms model collapse (smoke test shows !!!! output).')
        print('      Fix: re-run NB16 with LR=1e-4 and clean data from NB15.')
    elif ft_ov > b_ov + 0.02:
        print('  ✓  Fine-tuned model outperforms base on token overlap.')

if _overall_ft_syntax:
    ft_s = _avg(_overall_ft_syntax)
    b_s  = _avg(_overall_base_syntax)
    print(f'  Avg syntax pass:    FT={ft_s:.1%}  Base={b_s:.1%}  Δ={ft_s-b_s:+.1%}')

print(f'\nOutputs:')
print(f'  {EVAL_DIR / "report.json"}')
print(f'  {EVAL_DIR / "details.jsonl"}')
print('=' * 65)

notebook 18 — EVALUATION SUMMARY

Test set: 39 samples
  syntax_qa                : 24
  code_generation          : 14
  code_explanation         : 1

Metrics (fine-tuned vs base):
  Task                   Metric                             FT     Base    Delta
  ---------------------- ---------------------------- -------- -------- --------
  code_explanation       token_overlap (F1)              0.460    0.286   +0.174  ✓
  code_generation        syntax_pass_rate               35.7%    7.1%  +28.6%  ✓
  code_generation        hallucination_rate (↓)         30.8%   26.0%   +4.8%
  syntax_qa              token_overlap (F1)              0.368    0.189   +0.179  ✓
  syntax_qa              hallucination_rate (↓)         19.4%   24.1%   -4.7%

Overall:
  Avg token overlap:  FT=0.414  Base=0.237  Δ=+0.177
  ✓  Fine-tuned model outperforms base on token overlap.
  Avg syntax pass:    FT=35.7%  Base=7.1%  Δ=+28.6%

Outputs:
  /Users/kris/Projects/ARO/ARO-Train/Train/data/07_eval/report.json
  